In [ ]:
import warnings
warnings.filterwarnings('ignore')
warnings.simplefilter('ignore')

from datasets import load_dataset
from matplotlib import pyplot as plt

import random
import json
from PIL import Image
import pandas as pd
from tqdm import tqdm

import numpy as np

import torch
import torchvision

from utils import *

def choose_device() -> str:
    if torch.cuda.is_available():
        return "cuda:0"
    if hasattr(torch.backends, "mps"):
        if torch.backends.mps.is_available():
            return "mps"
    return "cpu"

device = torch.device(choose_device())
print(torch.cuda.get_device_name(device))

# fix seed for reproducibility
seed = 123
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed) # multi-GPU

In [ ]:
imagenet_transform = torchvision.transforms.Compose(
    [
        SquareCropAndResize(224),
        torchvision.transforms.ToTensor(),
        torchvision.transforms.Normalize(
            mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]
        ),
    ]
)

inverse_transform = torchvision.transforms.Compose(
    [
        torchvision.transforms.Normalize(
            mean=[0.0, 0.0, 0.0], std=[1 / 0.229, 1 / 0.224, 1 / 0.225]
        ),
        torchvision.transforms.Normalize(
            mean=[-0.485, -0.456, -0.406], std=[1.0, 1.0, 1.0]
        ),
    ]
)

with open("imagenet_class_index.json") as f:
    imagenet_labels = list(json.load(f).values())


### Load model

In [ ]:
resnet = torchvision.models.resnet50(weights="ResNet50_Weights.IMAGENET1K_V1")
resnet.eval().to(device)

# ImageNet-1k + ImageNet-S

You may need to accept the huggingface terms and conditions to download the [imagenet-1k](https://huggingface.co/datasets/imagenet-1k) dataset. Also, make sure you have run `huggingface-cli login`.

In [ ]:
imgent_ds = load_dataset("imagenet-1k", streaming=False, split="validation")

### Convert huggingface dataset to dataframe

In [ ]:
rows = []
for i, (img, l) in enumerate(zip(imgent_ds._data["image"], imgent_ds._data["label"])):
    l = l.as_py()
    path = img["path"].as_py()
    row = {}
    row["category"] = path.split("/")[-1].split("_")[-1].split(".")[0]
    row["filename"] = path.split("/")[-1].split("_n")[0]
    row["label_names"] = imagenet_labels[l]
    row["label"] = l
    row["dataset_idx"] = i
    rows.append(row)

# set dtype int instead of numpy.int64
imgent_df = pd.DataFrame(rows)
del rows
imgent_df.head()

### Map ImageNet-1k to ImageNet-S

Before mapping the ImageNet-1k to [ImageNet-S](https://github.com/LUSSeg/ImageNet-S), we need to download [ImageNetS919](https://github.com/LUSSeg/ImageNet-S/releases/download/ImageNet-S/ImageNetS919-5f7f58ae1003d21da9409a8576bf7680.zip). Make sure you have the right path to the ImageNetS919 folder.

In [ ]:
import glob

file_paths = glob.glob("ImageNetS919/validation-segmentation/*/*")
imgnet_s_path = [path.split("/")[-1].split(".")[0] for path in file_paths]
print(len(file_paths), len(imgnet_s_path))
imgnet_s_path[:5]

In [ ]:
# match names and dataframe
imgnet_s_df = imgent_df[imgent_df["filename"].isin(imgnet_s_path)].copy()
# match file_paths and df_seg['filename']
imgnet_s_df["file_path"] = imgnet_s_df["filename"].apply(
    lambda x: file_paths[imgnet_s_path.index(x)]
)

print(len(imgnet_s_df))
imgnet_s_df[['filename', 'file_path']].head()

Extract labels from segmentation masks (see [here](https://github.com/LUSSeg/ImageNet-S?tab=readme-ov-file#qa) for more details).

In [ ]:
def get_seg_id(path):
    segmentation = Image.open(path)  # RGB
    segmentation = np.array(segmentation)
    seg = segmentation[:, :, 1] * 256 + segmentation[:, :, 0]  # R+G*256
    seg = np.unique(seg)
    seg = seg[(seg != 0) & (seg != 1000)]
    return seg


imgnet_s_df["segmentation_id"] = imgnet_s_df["file_path"].apply(lambda x: get_seg_id(x))
imgnet_s_df.head()

In [ ]:
# filter out masks with multiple classes
imgnet_s_df = imgnet_s_df[imgnet_s_df["segmentation_id"].apply(lambda x: len(x)) == 1].copy()
print(len(imgnet_s_df))
imgnet_s_df.head()

### Saliency map visualization

In [ ]:
from utils import *

xai_methods = (
    # todo add your (SSL) methods here
    # as example, here is an SSL method
    # note that interpolation=None produces 7x7 maps
    # this is required for vector graphics
    ("ResNet Grad-CAM", GradCAMHeatmap(resnet, resnet.layer4, imagenet_labels, None)),
    ("Smooth Grad Campp", torchcam_SmoothGradCAMpp(resnet, resnet.layer4, imagenet_labels, None)),
    ("CAM", torchcam_CAM(resnet, resnet.layer4, imagenet_labels, None)),
    ("ScoreCAM", torchcam_ScoreCAM(resnet, resnet.layer4, imagenet_labels, None)),
    ("SSCAM", torchcam_SSCAM(resnet, resnet.layer4, imagenet_labels, None)),
    ("GradCAM", torchcam_GradCam(resnet, resnet.layer4, imagenet_labels, None)),
    ("GradCAMpp", torchcam_GradCampp(resnet, resnet.layer4, imagenet_labels, None)),
    ("XGradCAM", torchcam_XGradCAM(resnet, resnet.layer4, imagenet_labels, None)),
    ("LayerCAM", torchcam_LayerCAM(resnet, resnet.layer4, imagenet_labels, None)),
)

df_sample = imgnet_s_df.sample(3)


fig, axes = plt.subplots(
    nrows=len(df_sample),
    ncols=len(xai_methods) + 1,
    figsize=((len(xai_methods) + 1) * 2, 2 * len(df_sample)),
)
for ax in axes.flatten():
    ax.axis("off")

x = np.linspace(0, 224, 8)
y = np.linspace(0, 224, 8)
X, Y = np.meshgrid(x, y)

for i, row in enumerate(tqdm(df_sample.itertuples(), total=len(df_sample))):
    img = imgent_ds[int(row.dataset_idx)]['image']
    img = SquareCropAndResize(224)(img)
    img_tensor = imagenet_transform(img).unsqueeze(0).to(device)
   
    seg_img = Image.open(row.file_path)
    seg_img = SquareCropAndResize(224)(seg_img)
    seg_img = np.array(seg_img)

    mask = np.array(seg_img[:, :, 0]).astype(np.float32)

    mask[mask != 0] = 1
    mask_alpha = plt.cm.gray(mask)
    mask_alpha[..., 3] = mask

    if i == 0:
        axes[i, 0].title.set_text("Original image")

    axes[i, 0].imshow(img)

    # add vertical title left
    true_label = imagenet_labels[row.label]
    true_label = true_label.split(",")[0]
    axes[i, 0].text(
        -0.1,
        0.5,
        f"True: {true_label}",
        size=12,
        ha="center",
        va="center",
        rotation=90,
        transform=axes[i, 0].transAxes,
    )

    for j, method_info in enumerate(xai_methods):
        name, method = method_info
        heatmap, pred_label = method(img_tensor)
        if i == 0:
            axes[i, j + 1].title.set_text(name)
        heatmap = np.flipud(heatmap[0].cpu().numpy())
        axes[i, j + 1].pcolormesh(X, Y, heatmap, cmap='viridis', shading='auto')
        axes[i, j + 1].set_aspect('equal')
        axes[i, j + 1].contour(mask[::-1], levels=[0.5], colors='w', alpha=0.6)
        axes[i, j + 1].title.set_text(pred_label)

plt.tight_layout()

# PASCAL Segmentation


In [ ]:
pascal_labels = {
    1: "aeroplane",
    2: "bicycle",
    3: "bird",
    4: "boat",
    5: "bottle",
    6: "bus",
    7: "car",
    8: "cat",
    9: "chair",
    10: "cow",
    11: "diningtable",
    12: "dog",
    13: "horse",
    14: "motorbike",
    15: "person",
    16: "potted plant",
    17: "sheep",
    18: "sofa",
    19: "train",
    20: "tv/monitor",
}


pascal_ds = torchvision.datasets.VOCSegmentation(
    root="data/VOCdevkit",
    year="2012",
    image_set="val",
    download=True,
    transform=imagenet_transform,
    target_transform=SquareCropAndResize(224, interpolation=Image.NEAREST),
)



### Create pandas dataframe from torch dataset


In [ ]:
data = []

for idx, (_, mask) in enumerate(pascal_ds):
    mask = np.array(mask)
    labels = np.unique(mask)
    labels = labels[(labels != 0) & (labels != 255)]

    data.append(
        {
            "dataset_idx": idx,
            "labels": labels,
            "label_count": len(labels),
            "label_names": [pascal_labels[label] for label in labels],
        }
    )

# Create a pandas dataframe
pascal_df = pd.DataFrame(data)


print(f"Number of images: {len(pascal_df)}")

pascal_df["label_names"].explode().value_counts().plot(
    kind="bar", figsize=(6, 3), title="Pascal VOC 2012 - labels distribution"
)

In [ ]:
pascal_df = pascal_df[pascal_df["label_count"] == 1].copy()

exclude_classes = [
    "person",
    "tv/monitor",
    "sofa",
    "potted plant",
    "diningtable",
    "chair",
    "bottle",
]
pascal_df = pascal_df[
    ~pascal_df["label_names"].apply(lambda x: any(cls in x for cls in exclude_classes))
].copy()

print(f"Number of images: {len(pascal_df)}")

pascal_df["label_names"].explode().value_counts().plot(
    kind="bar", figsize=(6, 3), title="Pascal VOC 2012 - labels distribution"
)

### Saliency map visualization

In [ ]:
xai_methods = (
    # todo add your (SSL) methods here
    # as example, here is an SSL method
    # note that interpolation=None produces 7x7 maps
    # this is required for vector graphics
    ("ResNet Grad-CAM", GradCAMHeatmap(resnet, resnet.layer4, imagenet_labels, None)),
    ("Smooth Grad Campp", torchcam_SmoothGradCAMpp(resnet, resnet.layer4, imagenet_labels, None)),
    ("CAM", torchcam_CAM(resnet, resnet.layer4, imagenet_labels, None)),
    ("ScoreCAM", torchcam_ScoreCAM(resnet, resnet.layer4, imagenet_labels, None)),
    ("SSCAM", torchcam_SSCAM(resnet, resnet.layer4, imagenet_labels, None)),
    ("GradCAM", torchcam_GradCam(resnet, resnet.layer4, imagenet_labels, None)),
    ("GradCAMpp", torchcam_GradCampp(resnet, resnet.layer4, imagenet_labels, None)),
    ("XGradCAM", torchcam_XGradCAM(resnet, resnet.layer4, imagenet_labels, None)),
    ("LayerCAM", torchcam_LayerCAM(resnet, resnet.layer4, imagenet_labels, None)),
)


df_sample = pascal_df.sample(3)

fig, axes = plt.subplots(
    nrows=len(df_sample),
    ncols=len(xai_methods) + 2,
    figsize=((len(xai_methods) + 2) * 2, 2 * len(df_sample)),
)
for ax in axes.flatten():
    ax.axis("off")

x = np.linspace(0, 224, 8)
y = np.linspace(0, 224, 8)
X, Y = np.meshgrid(x, y)



for i, row in enumerate(tqdm(df_sample.itertuples(), total=len(df_sample))):
    img_tensor, seg_img = pascal_ds[row.dataset_idx]
    img = inverse_transform(img_tensor).permute(1, 2, 0).numpy()

    img_tensor = img_tensor.unsqueeze(0).to(device)

    seg = np.array(seg_img)
    # remove outline
    seg[seg == 255] = 0

    seg_classes = np.unique(seg)
    seg_classes = seg_classes[seg_classes != 0]
    seg_labels = [pascal_labels[i] for i in seg_classes]
    seg_labels = ", ".join(seg_labels)

    seg = np.array(seg_img)
    mask = np.zeros_like(seg).astype(np.float32)
    mask[seg == 255] = 1
    mask_alpha = plt.cm.gray(mask)
    mask_alpha[..., 3] = mask

    if i == 0:
        axes[i, 0].title.set_text("Original image")

    axes[i, 0].imshow(img, vmin=img.min(), vmax=img.max())


    # add vertical title left
    axes[i, 0].text(
        -0.1,
        0.5,
        f"True: {seg_labels}\nPred.: {pred_label}",
        size=12,
        ha="center",
        va="center",
        rotation=90,
        transform=axes[i, 0].transAxes,
    )

    for j, method_info in enumerate(xai_methods):
        name, method = method_info
        heatmap, pred_label = method(img_tensor)
        if i == 0:
            axes[i, j + 1].title.set_text(name)
        heatmap = np.flipud(heatmap[0].cpu().numpy())
        axes[i, j + 1].pcolormesh(X, Y, heatmap, cmap='viridis', shading='auto')
        axes[i, j + 1].set_aspect('equal')
        axes[i, j + 1].contour(mask[::-1], levels=[0.5], colors='w', alpha=0.6)
        axes[i, j + 1].title.set_text(pred_label)

plt.tight_layout()